# DX799 Week 9: Gradient Boosting
**Integrated Capstone Project:** VFX, Animation, and the Film Industry  
**Research question:** Can gradient boosting improve predictions of whether a movie earns at least twice its production budget?

In the last few weeks, I tested logistic regression, support vector machines, decision trees, random forests, and K-Nearest Neighbors. Random forest has been the strongest classification model so far, but I wanted to see whether gradient boosting could improve on that result. This week I will focus on the learning rate, number of estimators, tree depth, and regularization. I will keep the same movie dataset, predictors, and train/test split to make the comparisons as fair as possible.

## 1. Gradient boosting and how I will evaluate it
Gradient boosting creates a series of small decision trees. Each tree tries to improve on the errors made by the earlier trees. Unlike random forests, which average many trees, gradient boosting builds trees one after another. The **learning rate** controls how much each new tree changes the prediction, while **n_estimators** controls the number of trees. **max_depth** determines how complicated each tree can become. I will also evaluate **min_samples_leaf** and **subsample**, which can help prevent the model from fitting individual movies too closely.

I will tune parameters using three-fold stratified cross-validation on the training data only. ROC-AUC will be the main selection metric. Accuracy, precision, recall, F1, and average precision will help show the types of mistakes the model makes. The held-out test set will only be used after the models have been selected.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay, log_loss)
SEED=42
path=Path('movies.csv')
if not path.exists(): path=Path('/mnt/data/movies.csv')
raw=pd.read_csv(path)
print('Input file:',path)
print('Original data:',raw.shape)

Input file: movies.csv
Original data: (18905, 23)


## 2. Prepare the same financial-success sample
I am using the Week 8 filtering rules. A movie counts as financially successful if its worldwide revenue is at least twice its reported production budget. This does not mean the movie was actually profitable because marketing, distribution, and theater costs are not included.

For the predictors, I will use budget, runtime, release year, animation and genre flags, and release month. I am not using ratings, votes, popularity, or actual revenue as predictors because these can reveal information about a film after release. This gives the model a more realistic pre-release forecasting question.

In [2]:
df = raw.drop_duplicates(subset=['movie_id']).copy()
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
for col in ['budget','revenue','runtime']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.loc[(df.budget > 0) & (df.revenue > 0) & df.release_year.between(1980,2024)
            & df.runtime.between(40,240) & df.release_date.notna()].copy()
df['budget_m'] = df.budget / 1e6
df['revenue_m'] = df.revenue / 1e6
df['gross_to_budget'] = df.revenue / df.budget
df['success'] = (df.gross_to_budget >= 2).astype(int)
genres = df.genres.fillna('').str.split('|')
df['animation'] = genres.map(lambda g: int('Animation' in g))
df['release_month'] = df.release_date.dt.month
for genre in ['Action','Comedy','Drama','Horror']:
    df['genre_' + genre.lower()] = genres.map(lambda g: int(genre in g))
for month in range(2,13):
    df[f'month_{month}'] = (df.release_month == month).astype(int)
features = ['budget_m','runtime','release_year','animation','genre_action','genre_comedy',
            'genre_drama','genre_horror'] + [f'month_{i}' for i in range(2,13)]
tr, te = train_test_split(np.arange(len(df)), test_size=.2, random_state=SEED)
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
X_train, X_test = train[features], test[features]
y_train, y_test = train.success, test.success
print(f'Filtered movies: {len(df):,}, training: {len(train):,}, test: {len(test):,}')
print(f'Predictors: {len(features)}; animated films: {df.animation.sum():,}')
print(f'Success share: {df.success.mean():.1%} overall, {y_train.mean():.1%} training, {y_test.mean():.1%} testing')
print('Missing predictor values:', int(X_train.isna().sum().sum() + X_test.isna().sum().sum()))

Filtered movies: 7,079, training: 5,663, test: 1,416
Predictors: 19; animated films: 376
Success share: 43.6% overall, 43.7% training, 43.1% testing
Missing predictor values: 0


## 3. Exploratory data analysis
Before fitting the models, I want to look at the class balance and how budget relates to success. I expect budget to be important, but I do not expect budget by itself to tell us whether a movie will earn back twice its cost. The graph below also helps us see why the model may struggle with movies that share similar budgets but have very different outcomes.

In [3]:
fig,axs=plt.subplots(1,2,figsize=(11,4))
counts=df.success.value_counts().reindex([0,1]); axs[0].bar(['Below 2x','At least 2x'],counts.values)
axs[0].set(ylabel='Movie count',title='Financial success: class balance')
for lab,name in [(0,'Below 2x'),(1,'At least 2x')]:
    axs[1].hist(df.loc[df.success==lab,'budget_m'].clip(upper=250),bins=25,alpha=.65,label=name)
axs[1].set(xlabel='Budget ($M, capped at $250M for display)',ylabel='Movies',title='Budget by financial success')
axs[1].legend();plt.tight_layout();plt.show()

## 4. Establish a simple gradient boosting baseline
I will start with a small gradient boosting model using a moderate learning rate and shallow trees. This gives us a starting point before tuning. I will compare training and test ROC-AUC to see whether it is memorizing the training set.

In [4]:
def metrics(name,model):
    pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
    return {'Model':name,'ROC-AUC':roc_auc_score(y_test,prob),'Accuracy':accuracy_score(y_test,pred),
            'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),
            'F1':f1_score(y_test,pred,zero_division=0),
            'Average precision':average_precision_score(y_test,prob)}
base=GradientBoostingClassifier(n_estimators=100,learning_rate=.05,max_depth=2,random_state=SEED)
base.fit(X_train,y_train)
print('Baseline training ROC-AUC:',round(roc_auc_score(y_train,base.predict_proba(X_train)[:,1]),4))
print(pd.DataFrame([metrics('Baseline gradient boosting',base)]).round(4).to_string(index=False))

Baseline training ROC-AUC: 0.6765
                     Model  ROC-AUC  Accuracy  Precision  Recall     F1  Average precision
Baseline gradient boosting   0.6433    0.6095      0.609  0.2651 0.3694             0.5785


## 5. Tune learning rate, number of estimators, depth, and regularization
A low learning rate normally needs more trees, while deeper trees can learn more detailed patterns. However, adding trees and increasing depth also increases overfitting risk. I will search several reasonable combinations and let training-only cross-validation select a model. To control complexity, I am also testing the minimum number of movies required in each leaf and the share of training rows used per boosting stage. Using less than 100% of rows is called stochastic gradient boosting.

**Reproducibility note:** The original executed search below used **three-fold**, not five-fold, stratified cross-validation and tested eight randomly sampled configurations. The original numerical results are retained so they remain traceable to the completed run. The additional checks later in the notebook use five-fold validation to assess stability. I will not choose a new model using the test-set results.

In [5]:
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
params={'learning_rate':[.02,.05,.1], 'n_estimators':[50,100,150], 'max_depth':[1,2,3],
        'min_samples_leaf':[5,20,50], 'subsample':[.75,1.0]}
search=RandomizedSearchCV(GradientBoostingClassifier(random_state=SEED),params,n_iter=8,
                          scoring='roc_auc',cv=cv,random_state=SEED,n_jobs=1,
                          return_train_score=True,refit=True)
search.fit(X_train,y_train)
best=search.best_estimator_
print('Selected parameters:',search.best_params_)
print('Best training CV ROC-AUC:',round(search.best_score_,4))
results=pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
print(results[['param_learning_rate','param_n_estimators','param_max_depth',
               'param_min_samples_leaf','param_subsample','mean_train_score','mean_test_score']]
      .head(8).round(4).to_string(index=False))

Selected parameters: {'subsample': 0.75, 'n_estimators': 100, 'min_samples_leaf': 50, 'max_depth': 3, 'learning_rate': 0.1}
Best training CV ROC-AUC: 0.6548
 param_learning_rate  param_n_estimators  param_max_depth  param_min_samples_leaf  param_subsample  mean_train_score  mean_test_score
                0.10                 100                3                      50             0.75            0.7345           0.6548
                0.05                 150                3                       5             0.75            0.7396           0.6539
                0.05                 150                3                      20             1.00            0.7305           0.6502
                0.10                 150                2                       5             1.00            0.7101           0.6471
                0.02                 100                3                      50             1.00            0.6800           0.6359
                0.02                 15

## 5A. What each hyperparameter means for this movie dataset

| Parameter | Meaning | Why it matters here |
|---|---|---|
| `learning_rate` | Shrinks each new tree's contribution | Slower updates can reduce the chance of fitting unusual films too aggressively, but may need more trees. |
| `n_estimators` | Number of boosting stages | More trees can capture complex relationships between movie characteristics and success, but may increase overfitting. |
| `max_depth` | Maximum depth of each individual decision tree | Deeper trees can capture more complicated interactions, such as whether budget behaves differently for animation and other genres. |
| `min_samples_leaf` | Minimum movies in a terminal leaf | Larger leaves prevent rules from being based on very small groups of movies. |
| `subsample` | Fraction of the training data used to fit each boosting stage | A value under 1.0 introduces randomness and can reduce variance. |

The original search selected a learning rate of 0.10, 100 trees, depth 3, 50 movies per leaf, and subsample 0.75. Its mean cross-validation ROC-AUC was **0.6548** versus a mean training-fold ROC-AUC of **0.7345**. The gap of about 0.080 indicates that the model fit its training folds better than its held-out validation folds. This is why I do not want to rely on training performance alone.

## 6. How learning rate and number of trees change results
To better understand the tradeoff, I will fit a few fixed learning-rate configurations with different numbers of trees. These are training-only cross-validation results, not an additional selection based on the test set. The plot can help show why a smaller learning rate may need more estimators.

In [6]:
rates=[.03,.1]; num_trees=[50,120]
rate_rows=[]
for r in rates:
    for n in num_trees:
        m=GradientBoostingClassifier(learning_rate=r,n_estimators=n,max_depth=2,
             min_samples_leaf=20,subsample=.8,random_state=SEED)
        s=cross_validate(m,X_train,y_train,cv=cv,scoring='roc_auc',n_jobs=1)
        rate_rows.append({'learning_rate':r,'n_estimators':n,'CV ROC-AUC':s['test_score'].mean()})
rates_df=pd.DataFrame(rate_rows)
fig,ax=plt.subplots(figsize=(7.6,4.5))
for r in rates:
    d=rates_df[rates_df.learning_rate==r]
    ax.plot(d.n_estimators,d['CV ROC-AUC'],marker='o',label=f'Learning rate {r}')
ax.set(xlabel='Number of estimators',ylabel='Training CV ROC-AUC',title='Learning rate vs. number of estimators')
ax.legend();ax.grid(alpha=.25);plt.tight_layout();plt.show()
print(rates_df.round(4).to_string(index=False))

/tmp/ipykernel_1158/2978662637.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend();ax.grid(alpha=.25);plt.tight_layout();plt.show()


## 7. Regularization and overfitting
Gradient boosting can overfit if its trees become too deep or if it adds too many stages. I will compare a more flexible model to the tuned model. I will also plot the tuned model's training and held-out ROC-AUC at different boosting stages. The test-set curve is a diagnostic only. It will not be used to choose the number of trees because that would leak information from the test set into model selection.

In [7]:
flex=GradientBoostingClassifier(n_estimators=120,learning_rate=.1,max_depth=4,
                                    min_samples_leaf=1,subsample=1,random_state=SEED)
flex.fit(X_train,y_train)
for label,m in [('Tuned gradient boosting',best),('Flexible comparison',flex)]:
    train_auc=roc_auc_score(y_train,m.predict_proba(X_train)[:,1])
    test_auc=roc_auc_score(y_test,m.predict_proba(X_test)[:,1])
    print(f'{label}: train ROC-AUC={train_auc:.4f}, held-out ROC-AUC={test_auc:.4f}, gap={train_auc-test_auc:.4f}')
steps=[]
for i,(train_prob,test_prob) in enumerate(zip(best.staged_predict_proba(X_train),best.staged_predict_proba(X_test)),1):
    steps.append((i,roc_auc_score(y_train,train_prob[:,1]),roc_auc_score(y_test,test_prob[:,1])))
steps=np.array(steps)
fig,ax=plt.subplots(figsize=(8,4.4))
ax.plot(steps[:,0],steps[:,1],label='Training')
ax.plot(steps[:,0],steps[:,2],label='Held-out test (diagnostic only)')
ax.set(xlabel='Boosting stage',ylabel='ROC-AUC',title='Training and test learning curves')
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()

Tuned gradient boosting: train ROC-AUC=0.7215, held-out ROC-AUC=0.6643, gap=0.0572
Flexible comparison: train ROC-AUC=0.7894, held-out ROC-AUC=0.6579, gap=0.1316


## 7A. Quantify generalization and check cross-validation stability

For my Milestone 1 paper, I lost points because I did not explain overfitting in enough detail. Here I will calculate the training, cross-validation, and test ROC-AUC separately for the previously selected model. I will also perform a five-fold **validation check** of the already selected configuration. This check is not a second hyperparameter search and will not be used to choose parameters after looking at the test set.

The original three-fold search was selected using only training data. In a rigorous future comparison, nested cross-validation or a dedicated validation set would give an even less optimistic estimate of the model-selection procedure. The five-fold result below is a diagnostic stability check, not an independent unbiased estimate of the selection process.

In [8]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
stability = cross_validate(
    GradientBoostingClassifier(**search.best_params_, random_state=SEED),
    X_train, y_train, cv=cv5, scoring='roc_auc', return_train_score=True, n_jobs=1
)
stability_table = pd.DataFrame({
    'Fold': np.arange(1, 6),
    'Training ROC-AUC': stability['train_score'],
    'Validation ROC-AUC': stability['test_score'],
})
stability_table['Generalization gap'] = (stability_table['Training ROC-AUC'] -
                                         stability_table['Validation ROC-AUC'])
print(stability_table.round(4).to_string(index=False))
print('Mean validation ROC-AUC:', round(stability_table['Validation ROC-AUC'].mean(), 4))
print('Validation ROC-AUC standard deviation:',
      round(stability_table['Validation ROC-AUC'].std(ddof=1), 4))
print('Mean training-validation gap:',
      round(stability_table['Generalization gap'].mean(), 4))

 Fold  Training ROC-AUC  Validation ROC-AUC  Generalization gap
    1            0.7324              0.6253              0.1072
    2            0.7261              0.6761              0.0500
    3            0.7266              0.6671              0.0594
    4            0.7244              0.6555              0.0689
    5            0.7237              0.6806              0.0431
Mean validation ROC-AUC: 0.6609
Validation ROC-AUC standard deviation: 0.0221
Mean training-validation gap: 0.0657


## 7B. Compare model complexity without using the test set for selection

An intentionally more flexible model allows smaller leaves, grows deeper trees, and uses the entire sample at every stage. I expect it to fit training movies better but potentially perform worse on unseen movies. In my original run, the tuned model's training/test ROC-AUC values were **0.7215 / 0.6643**, while the flexible model's were **0.7894 / 0.6579**. The test-set difference is small, but the much larger training-to-test gap of the flexible model (0.1316 compared with 0.0572) is consistent with greater overfitting.

The following chart uses cross-validation on the training set to investigate the tradeoff without choosing model settings based on test performance.

In [9]:
complexity_models = {
    'Selected regularized': GradientBoostingClassifier(**search.best_params_, random_state=SEED),
    'Flexible': GradientBoostingClassifier(n_estimators=120, learning_rate=.1,
        max_depth=4, min_samples_leaf=1, subsample=1.0, random_state=SEED)
}
complexity_rows = []
for label, model in complexity_models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv5, scoring='roc_auc',
                            return_train_score=True, n_jobs=1)
    complexity_rows.append({
        'Model': label,
        'Mean train ROC-AUC': scores['train_score'].mean(),
        'Mean validation ROC-AUC': scores['test_score'].mean(),
        'Train-validation gap': scores['train_score'].mean() - scores['test_score'].mean()
    })
complexity_table = pd.DataFrame(complexity_rows)
print(complexity_table.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 4.4))
positions = np.arange(len(complexity_table)); width = .34
ax.bar(positions-width/2, complexity_table['Mean train ROC-AUC'], width, label='Training')
ax.bar(positions+width/2, complexity_table['Mean validation ROC-AUC'], width, label='Validation')
ax.set_xticks(positions, complexity_table['Model'])
ax.set(ylabel='Mean 5-fold ROC-AUC', title='Regularization and generalization')
ax.legend(); plt.tight_layout(); plt.show()

               Model  Mean train ROC-AUC  Mean validation ROC-AUC  Train-validation gap
Selected regularized              0.7266                   0.6609                0.0657
            Flexible              0.8070                   0.6504                0.1567


/tmp/ipykernel_1158/3552181922.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.tight_layout(); plt.show()


## 8. Compare with random forest and the Week 8 KNN result
I will re-fit a random forest using a fixed configuration as a comparison on the same split. I will also refer to the previously reported Week 8 KNN result. The random forest below is a fresh fit rather than an exact recreation of the Week 6 hyperparameter search, so these two results should not be treated as interchangeable. The main goal is to determine whether boosting shows a meaningful advantage over another tree-based approach.

In [10]:
forest=RandomForestClassifier(n_estimators=120,min_samples_leaf=5,max_features='sqrt',
                                    random_state=SEED,n_jobs=1)
forest.fit(X_train,y_train)
comparison=pd.DataFrame([metrics('Gradient boosting baseline',base),
                          metrics('Tuned gradient boosting',best),
                          metrics('Random forest comparison',forest)]).set_index('Model')
print(comparison.round(4).to_string())
print('Historical Week 8 scaled KNN test ROC-AUC: approximately 0.615')
print('Historical Week 6 random forest test ROC-AUC: approximately 0.651')

                            ROC-AUC  Accuracy  Precision  Recall      F1  Average precision
Model                                                                                      
Gradient boosting baseline   0.6433    0.6095     0.6090  0.2651  0.3694             0.5785
Tuned gradient boosting      0.6643    0.6229     0.5937  0.3993  0.4775             0.5934
Random forest comparison     0.6510    0.6271     0.5986  0.4124  0.4884             0.5890
Historical Week 8 scaled KNN test ROC-AUC: approximately 0.615
Historical Week 6 random forest test ROC-AUC: approximately 0.651


## 9. ROC curves, confusion matrix, and feature importance
These plots help show the difference between the models beyond a single score. I will also examine feature importance to see which variables the gradient boosting model relied on most. These are predictive associations, not proof that a particular movie feature causes higher revenue.

In [11]:
fig,axs=plt.subplots(1,2,figsize=(11.5,4.4))
for label,m in [('Gradient boosting',best),('Random forest',forest)]:
    RocCurveDisplay.from_estimator(m,X_test,y_test,ax=axs[0],name=label)
    PrecisionRecallDisplay.from_estimator(m,X_test,y_test,ax=axs[1],name=label)
axs[0].set_title('Held-out ROC curves');axs[1].set_title('Held-out precision-recall curves')
plt.tight_layout();plt.show()
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
ConfusionMatrixDisplay.from_estimator(best,X_test,y_test,display_labels=['Below 2x','At least 2x'],
                                     ax=axs[0],colorbar=False,cmap='Blues')
axs[0].set_title('Tuned gradient boosting')
imp=pd.Series(best.feature_importances_,index=features).sort_values().tail(10)
axs[1].barh(imp.index,imp.values);axs[1].set(xlabel='Impurity-based importance',title='Top ten predictive features')
plt.tight_layout();plt.show()

/tmp/ipykernel_1158/3557574638.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()
/tmp/ipykernel_1158/3557574638.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.show()


## 9A. Translate the results into real-world prediction errors

ROC-AUC measures ranking ability across possible classification thresholds. However, a studio deciding which projects to investigate might care more about the number of successful films overlooked or the false alarms it receives. The confusion matrix and recall answer a different practical question from ROC-AUC. In the original run, the tuned gradient boosting model produced an F1 of **0.4775** and recall of **0.3993**. That means it identified roughly 40% of the films in the held-out sample that exceeded the two-times-budget threshold, while missing roughly 60% at the default cutoff.

A threshold can change this balance, but it must be selected with training-only cross-validation and a clearly stated decision objective rather than selected because it happens to improve test results. The table below reports the held-out prediction counts at the default 0.50 threshold for transparency.

In [12]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, brier_score_loss
p_test = best.predict_proba(X_test)[:, 1]
yhat = (p_test >= 0.50).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, yhat, labels=[0, 1]).ravel()
print(pd.DataFrame([{
    'Correctly rejected (TN)': int(tn), 'False alarms (FP)': int(fp),
    'Successful films missed (FN)': int(fn), 'Successful films found (TP)': int(tp),
    'Balanced accuracy': balanced_accuracy_score(y_test, yhat),
    'Brier score': brier_score_loss(y_test, p_test)
}]).round(4).to_string(index=False))
print(f'At the fixed 0.50 cutoff, {fn} of {fn + tp} successful films were missed.')
print('These numbers describe classification, not film profitability or causal effects.')

 Correctly rejected (TN)  False alarms (FP)  Successful films missed (FN)  Successful films found (TP)  Balanced accuracy  Brier score
                     638                167                           367                          244             0.5959       0.2254
At the fixed 0.50 cutoff, 367 of 611 successful films were missed.
These numbers describe classification, not film profitability or causal effects.


## 9B. Inspect whether budget importance reflects useful discrimination

Feature importance in a tree ensemble is not a causal effect. It can favor features that offer many possible split points, and correlated movie characteristics can share predictive information. The permutation-importance calculation below measures how much ROC-AUC decreases when one feature is shuffled on the **training data**. This is a diagnostic of the fitted model rather than an additional basis for tuning against the test set.

In [13]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(best, X_train, y_train, scoring='roc_auc',
                              n_repeats=5, random_state=SEED, n_jobs=1)
perm_df = pd.DataFrame({'Feature': features,
                        'Mean ROC-AUC decrease': perm.importances_mean,
                        'Std across shuffles': perm.importances_std})
print(perm_df.sort_values('Mean ROC-AUC decrease', ascending=False)
      .head(12).round(4).to_string(index=False))
top = perm_df.nlargest(10, 'Mean ROC-AUC decrease').sort_values('Mean ROC-AUC decrease')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top['Feature'], top['Mean ROC-AUC decrease'])
ax.set(xlabel='Mean training ROC-AUC decrease after permutation',
       title='Permutation feature importance (training diagnostic)')
plt.tight_layout(); plt.show()

     Feature  Mean ROC-AUC decrease  Std across shuffles
    budget_m                 0.0844               0.0019
release_year                 0.0672               0.0028
     runtime                 0.0547               0.0049
 genre_drama                 0.0276               0.0035
genre_horror                 0.0174               0.0007
genre_comedy                 0.0159               0.0013
genre_action                 0.0120               0.0011
     month_9                 0.0100               0.0020
    month_10                 0.0071               0.0006
     month_6                 0.0062               0.0010
     month_8                 0.0024               0.0009
     month_7                 0.0023               0.0004


/tmp/ipykernel_1158/1721851023.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 10. Conclusions for my Capstone
This week helped me see how different gradient boosting settings can change the results. In particular, learning rate and number of trees need to be considered together. Tree depth, minimum leaf size, and subsampling can help reduce overfitting, but there is no guarantee that a more complicated model will predict new movies more accurately. The results below summarize the actual held-out scores so that I can use them accurately when writing Milestone Two.

In [14]:
b=comparison.loc['Tuned gradient boosting']; rf=comparison.loc['Random forest comparison']
print(f'The final sample contained {len(df):,} movies with {len(features)} pre-release predictors.')
print(f'Tuned gradient boosting achieved ROC-AUC {b["ROC-AUC"]:.3f}, accuracy {b["Accuracy"]:.1%}, and F1 {b["F1"]:.3f}.')
print(f'The random forest comparison achieved ROC-AUC {rf["ROC-AUC"]:.3f} and F1 {rf["F1"]:.3f}.')
print(f'Training-only cross-validation chose {search.best_params_}.')
print('Limitations: financial values may be incomplete, the 2x-budget target is not a profitability measure, and important predictors such as marketing and franchise strength are not available.')

The final sample contained 7,079 movies with 19 pre-release predictors.
Tuned gradient boosting achieved ROC-AUC 0.664, accuracy 62.3%, and F1 0.477.
The random forest comparison achieved ROC-AUC 0.651 and F1 0.488.
Training-only cross-validation chose {'subsample': 0.75, 'n_estimators': 100, 'min_samples_leaf': 50, 'max_depth': 3, 'learning_rate': 0.1}.
Limitations: financial values may be incomplete, the 2x-budget target is not a profitability measure, and important predictors such as marketing and franchise strength are not available.


## 10A. Evidence-based conclusions to carry into Milestone 2

**What was expected?** I expected boosting to outperform some of the earlier classifiers because it builds a sequence of trees that gradually improves the predictions. The tuned model achieved test ROC-AUC **0.6643**, compared with **0.615** from the Week 8 KNN notebook on the same classification question.

**What was surprising?** Gradient boosting did not win on every metric. The random forest comparison had an F1 of **0.4884**, slightly higher than boosting's **0.4775**, despite boosting having the higher ROC-AUC (0.6643 versus 0.6510). This matters because a model can rank movies better overall while still making less favorable decisions at a specific classification threshold.

**Why regularization mattered:** The deeper, less constrained comparison achieved a higher training ROC-AUC but a slightly worse test ROC-AUC. The stronger training fit was not evidence of superior prediction. A moderate tree depth, larger minimum leaf, and subsampling helped control complexity in the selected configuration.

**What the results do not prove:** The two-times-budget label is a revenue-based proxy, not a measure of profit; marketing and distribution expenses are missing. The data exclude films without reported positive budgets and revenues. Movies also change over time, so a random train/test split is not the same as testing on genuinely future releases. Finally, the original randomized search only evaluated eight configurations, meaning the selected settings are the best among those sampled, not necessarily globally optimal.

**Milestone 2 focus:** Week 9 is a good deep-dive topic because it provides a direct quantitative comparison of baseline versus tuned boosting, learning rate and tree count, regularization, generalization gaps, and two different definitions of model quality (ROC-AUC versus F1). The new diagnostic cells above should be run before using their outputs in the paper.